In [1]:
from tqdm import tqdm
import pandas as pd
import random
import json

In [2]:
def read_jsonl(path):
    data = []
    with open(path, 'r') as f:
        for line in f:
            item = json.loads(line.strip())
            data.append(item)
    return data

TRAIN DATASET

In [ ]:
# This code works for the course "Digital Signal Theory" from Kyushu University used in the experiments
# For a new course, you need to include here the information about the lecture materials included in the course (contentsid)
# the number of pages of each material (pages) and the name of the corresponding class (title)

course_id = 207
course_materials = pd.read_csv("./pid{}_material_info.csv".format(course_id))[["contentsid","pages","title"]]


non_informative = ["TITLE_SLIDE","AGENDA_SLIDE","HEADER_SLIDE","CONCLUSION_SLIDE"]

all_indices = []
slide_texts = []

for n_material in tqdm(range(course_materials.shape[0])):
    material_id = course_materials["contentsid"][n_material]
    max_pages = course_materials["pages"][n_material]
    title = course_materials["title"][n_material]

    slide_types = []
    # Metadata generated in the previous notebook EXTRACT_TEXT_AND_LABELS
    # Specially useful for lecture slides, where some slides are not informative
    with open(f"./Agent_resources/Slide_types/pid_{course_id}_{material_id}_slide_types.txt", "r") as file:
        for line in file:
            slide_types.append(line.strip())

    for i in range(1,max_pages+1):
        if not slide_types[i-1] in non_informative:
            for j in range(1,4):
                # Metadata generated in the previous notebook EXTRACT_TITLE_AND_CONTENTS
                with open("./Agent_resources/TitleC_versions/pid{}_{}_title_{}_p{}.txt".format(course_id,material_id,j,i), "r", encoding="utf-8") as f:
                    crrn_title = f.read()
                with open("./Agent_resources/TitleC_versions/pid{}_{}_body_{}_p{}.txt".format(course_id,material_id,j,i), "r", encoding="utf-8") as f:
                    crrn_body = f.read()
                for k in range(2):
                    all_indices.append({"material_id":material_id,"page":i,"version":j,"title":k})
                    if k==0:
                        slide_texts.append(crrn_body)
                    else:
                        slide_texts.append(crrn_title + ":\n" + crrn_body)

100%|██████████| 12/12 [00:00<00:00, 71.84it/s]


In [4]:
slide_index_dictionary = pd.DataFrame(all_indices)
slide_index_dictionary = slide_index_dictionary.reset_index(drop=False)

In [ ]:
queries = []
samples = []
query_idx = 0

for n_material in tqdm(range(course_materials.shape[0])):
    material_id = course_materials["contentsid"][n_material]
    max_pages = course_materials["pages"][n_material]
    title = course_materials["title"][n_material]

    slide_types = []
    # Metadata generated in the previous notebook EXTRACT_TEXT_AND_LABELS
    # Specially useful for lecture slides, where some slides are not informative
    with open(f"./Agent_resources/Slide_types/pid_{course_id}_{material_id}_slide_types.txt", "r") as file:
        for line in file:
            slide_types.append(line.strip())

    non_informative = ["TITLE_SLIDE","AGENDA_SLIDE","HEADER_SLIDE","CONCLUSION_SLIDE"]
    slide_filtered = []
    slide_possible = []

    for i in range(1,max_pages+1):
        if slide_types[i-1] in non_informative:
            slide_filtered.append(i)
        else:
            slide_possible.append(i)

    # 10: Number of generation samples (each generation may find different characteristics) 
    # in the previous notebook THE_PROMPT_ENGINEERING
    # If this parameter was changed there, it should be also changed here
    for q_sample in range(10):
        try:
            questions = read_jsonl(f'./Agent_resources/Questions/pid{course_id}_{material_id}_QA_{q_sample}.jsonl')
        except:
            raise NameError(f"Error of file name: {q_sample} sample.")
            continue
        
        for question in questions:
            responses = list(set(question['Answer_info']) - set(slide_filtered))
            responses_similar = list(set(question['Complement_info']) - set(slide_filtered))
            responses_negative = list(set(slide_possible)-(set(responses) | set(responses_similar)))

            set_neg = []
            for rn in responses_negative:
                ans_neg = slide_index_dictionary.loc[
                                (slide_index_dictionary["material_id"]==material_id)&
                                (slide_index_dictionary["page"]==rn)&
                                (slide_index_dictionary["title"]==1)]
                set_ans_neg = ans_neg["index"].to_list()
                set_neg.extend(set_ans_neg)

            if len(responses)>0:
                for r in responses:
                    ans = slide_index_dictionary.loc[
                        (slide_index_dictionary["material_id"]==material_id)&
                        (slide_index_dictionary["page"]==r)&
                        (slide_index_dictionary["title"]==1)]
                    set_ans = ans["index"].to_list()

                    if len(responses_similar)>0:
                        for rs in responses_similar:
                            ans_sim = slide_index_dictionary.loc[
                                (slide_index_dictionary["material_id"]==material_id)&
                                (slide_index_dictionary["page"]==rs)&
                                (slide_index_dictionary["title"]==1)]
                            set_ans_sim = ans_sim["index"].to_list()

                            samples.append([query_idx,set_ans,set_ans_sim,set_neg])
                            samples.append([query_idx+1,set_ans,set_ans_sim,set_neg])
                    else:
                        rs = random.choice(responses_negative)
                        ans_sim = slide_index_dictionary.loc[
                                (slide_index_dictionary["material_id"]==material_id)&
                                (slide_index_dictionary["page"]==rs)&
                                (slide_index_dictionary["title"]==1)]
                        set_ans_sim = ans_sim["index"].to_list()

                        samples.append([query_idx,set_ans,set_ans_sim,set_neg])
                        samples.append([query_idx+1,set_ans,set_ans_sim,set_neg])

            queries.append("Question: " + question['Question'])
            queries.append("Answer: " + question['Answer'])
            query_idx += 2

In [ ]:
for n_material in tqdm(range(course_materials.shape[0])):
    material_id = course_materials["contentsid"][n_material]
    max_pages = course_materials["pages"][n_material]
    title = course_materials["title"][n_material]

    slide_types = []
    # Metadata generated in the previous notebook EXTRACT_TEXT_AND_LABELS
    # Specially useful for lecture slides, where some slides are not informative
    with open(f"./Agent_resources/Slide_types/pid_{course_id}_{material_id}_slide_types.txt", "r") as file:
        for line in file:
            slide_types.append(line.strip())

    non_informative = ["TITLE_SLIDE","AGENDA_SLIDE","HEADER_SLIDE","CONCLUSION_SLIDE"]
    slide_filtered = []
    slide_possible = []

    for i in range(1,max_pages+1):
        if slide_types[i-1] in non_informative:
            slide_filtered.append(i)
        else:
            slide_possible.append(i)

    # 10: Number of generation samples (each generation may find different characteristics) 
    # in the previous notebook THE_PROMPT_ENGINEERING
    # If this parameter was changed there, it should be also changed here
    for q_sample in range(10):
        try:
            importances = read_jsonl(f'./Agent_resources/Importances/pid{course_id}_{material_id}_IMPORTANCE_{q_sample}.jsonl')
            importances = pd.DataFrame(importances)
        except:
            raise NameError(f"Error of file name: {q_sample} sample.")
            continue

        for slide_page in slide_possible:
            if importances.loc[importances["Slide"]==slide_page].shape[0]==0:
                print(f"Error, slide not included: {slide_page}")
                continue

            responses = [slide_page]
            responses_similar = list(set(importances.loc[importances["Slide"]==slide_page]["Ref_slides"].tolist()[0]) - set(slide_filtered))
            responses_negative = list(set(slide_possible)-(set(responses) | set(responses_similar)))

            set_neg_non_title = []
            set_neg_title = []
            for rn in responses_negative:
                ans_neg = slide_index_dictionary.loc[
                                (slide_index_dictionary["material_id"]==material_id)&
                                (slide_index_dictionary["page"]==rn)&
                                (slide_index_dictionary["title"]==0)]
                set_ans_neg = ans_neg["index"].to_list()
                set_neg_non_title.extend(set_ans_neg)

                ans_neg = slide_index_dictionary.loc[
                                (slide_index_dictionary["material_id"]==material_id)&
                                (slide_index_dictionary["page"]==rn)&
                                (slide_index_dictionary["title"]==1)]
                set_ans_neg = ans_neg["index"].to_list()
                set_neg_title.extend(set_ans_neg)
            
            ans = slide_index_dictionary.loc[
                (slide_index_dictionary["material_id"]==material_id)&
                (slide_index_dictionary["page"]==slide_page)&
                (slide_index_dictionary["title"]==0)]
            set_ans_non_title = ans["index"].to_list()

            ans = slide_index_dictionary.loc[
                (slide_index_dictionary["material_id"]==material_id)&
                (slide_index_dictionary["page"]==slide_page)&
                (slide_index_dictionary["title"]==1)]
            set_ans_title = ans["index"].to_list()

            if len(responses_similar)>0:
                for rs in responses_similar:
                    ans_sim = slide_index_dictionary.loc[
                        (slide_index_dictionary["material_id"]==material_id)&
                        (slide_index_dictionary["page"]==rs)&
                        (slide_index_dictionary["title"]==0)]
                    set_ans_sim_non_title = ans_sim["index"].to_list()

                    ans_sim = slide_index_dictionary.loc[
                        (slide_index_dictionary["material_id"]==material_id)&
                        (slide_index_dictionary["page"]==rs)&
                        (slide_index_dictionary["title"]==1)]
                    set_ans_sim_title = ans_sim["index"].to_list()

                    samples.append([query_idx,set_ans_non_title,set_ans_sim_non_title,set_neg_non_title])
                    samples.append([query_idx+1,set_ans_title,set_ans_sim_title,set_neg_title])
                    samples.append([query_idx+2,set_ans_title,set_ans_sim_title,set_neg_title])
            else:
                rs = random.choice(responses_negative)
                ans_sim = slide_index_dictionary.loc[
                        (slide_index_dictionary["material_id"]==material_id)&
                        (slide_index_dictionary["page"]==rs)&
                        (slide_index_dictionary["title"]==0)]
                set_ans_sim_non_title = ans_sim["index"].to_list()

                ans_sim = slide_index_dictionary.loc[
                    (slide_index_dictionary["material_id"]==material_id)&
                    (slide_index_dictionary["page"]==rs)&
                    (slide_index_dictionary["title"]==1)]
                set_ans_sim_title = ans_sim["index"].to_list()

                samples.append([query_idx,set_ans_non_title,set_ans_sim_non_title,set_neg_non_title])
                samples.append([query_idx+1,set_ans_title,set_ans_sim_title,set_neg_title])
                samples.append([query_idx+2,set_ans_title,set_ans_sim_title,set_neg_title])


            queries.append("Title: " + importances.loc[importances["Slide"]==slide_page]["Title"].tolist()[0])
            queries.append("Importance: " + importances.loc[importances["Slide"]==slide_page]["Importance"].tolist()[0])
            queries.append("Key Topics: " + importances.loc[importances["Slide"]==slide_page]["Key_topics"].tolist()[0])
            query_idx += 3
        

In [ ]:
with open(f"./Agent_resources/Final_finetuning/pid_{course_id}_train_queries.json", "w", encoding="utf-8") as file:
    json.dump(queries, file)

In [ ]:
with open(f"./Agent_resources/Final_finetuning/pid_{course_id}_train_slides.json", "w", encoding="utf-8") as file:
    json.dump(slide_texts, file)

In [ ]:
with open(f"./Agent_resources/Final_finetuning/pid_{course_id}_dataset_train.json", "w", encoding="utf-8") as file:
    json.dump(samples, file)

TEST DATASET

In [ ]:
course_id = 207
course_materials = pd.read_csv("./pid{}_material_info.csv".format(course_id))[["contentsid","pages","title"]]

non_informative = ["TITLE_SLIDE","AGENDA_SLIDE","HEADER_SLIDE","CONCLUSION_SLIDE"]

all_indices = []
slide_texts = []

for n_material in tqdm(range(course_materials.shape[0])):
    material_id = course_materials["contentsid"][n_material]
    max_pages = course_materials["pages"][n_material]
    title = course_materials["title"][n_material]

    slide_types = []
    with open(f"./Agent_resources/Slide_types/pid_{course_id}_{material_id}_slide_types.txt", "r") as file:
        for line in file:
            slide_types.append(line.strip())

    for i in range(1,max_pages+1):
        if not slide_types[i-1] in non_informative:
            j = 4
            with open("./Agent_resources/TitleC_versions/pid{}_{}_title_{}_p{}.txt".format(course_id,material_id,j,i), "r", encoding="utf-8") as f:
                crrn_title = f.read()
            with open("./Agent_resources/TitleC_versions/pid{}_{}_body_{}_p{}.txt".format(course_id,material_id,j,i), "r", encoding="utf-8") as f:
                crrn_body = f.read()
            for k in range(2):
                all_indices.append({"material_id":material_id,"page":i,"version":j,"title":k})
                if k==0:
                    slide_texts.append(crrn_body)
                else:
                    slide_texts.append(crrn_title + ":\n" + crrn_body)

In [ ]:
slide_index_dictionary = pd.DataFrame(all_indices)
slide_index_dictionary = slide_index_dictionary.reset_index(drop=False)

In [ ]:
queries = []
samples = []
query_idx = 0

for n_material in tqdm(range(course_materials.shape[0])):
    material_id = course_materials["contentsid"][n_material]
    max_pages = course_materials["pages"][n_material]
    title = course_materials["title"][n_material]

    slide_types = []
    with open(f"./Agent_resources/Slide_types/pid_{course_id}_{material_id}_slide_types.txt", "r") as file:
        for line in file:
            slide_types.append(line.strip())

    non_informative = ["TITLE_SLIDE","AGENDA_SLIDE","HEADER_SLIDE","CONCLUSION_SLIDE"]
    slide_filtered = []
    slide_possible = []

    for i in range(1,max_pages+1):
        if slide_types[i-1] in non_informative:
            slide_filtered.append(i)
        else:
            slide_possible.append(i)

    for q_sample in range(10):
        try:
            questions = read_jsonl(f'./Agent_resources/Questions/pid{course_id}_{material_id}_QA_{q_sample}_test.jsonl')
        except:
            raise NameError(f"Error of file name: {q_sample} sample.")
            continue
        
        for question in questions:
            responses = list(set(question['Answer_info']) - set(slide_filtered))
            responses_similar = list(set(question['Complement_info']) - set(slide_filtered))
            responses_negative = list(set(slide_possible)-(set(responses) | set(responses_similar)))

            set_neg = []
            for rn in responses_negative:
                ans_neg = slide_index_dictionary.loc[
                                (slide_index_dictionary["material_id"]==material_id)&
                                (slide_index_dictionary["page"]==rn)&
                                (slide_index_dictionary["title"]==1)]
                set_ans_neg = ans_neg["index"].to_list()
                set_neg.extend(set_ans_neg)

            if len(responses)>0:
                for r in responses:
                    ans = slide_index_dictionary.loc[
                        (slide_index_dictionary["material_id"]==material_id)&
                        (slide_index_dictionary["page"]==r)&
                        (slide_index_dictionary["title"]==1)]
                    set_ans = ans["index"].to_list()

                    if len(responses_similar)>0:
                        for rs in responses_similar:
                            ans_sim = slide_index_dictionary.loc[
                                (slide_index_dictionary["material_id"]==material_id)&
                                (slide_index_dictionary["page"]==rs)&
                                (slide_index_dictionary["title"]==1)]
                            set_ans_sim = ans_sim["index"].to_list()

                            ans_neg_choice = [random.choice(set_neg)]

                            samples.append([query_idx,set_ans,set_ans_sim,ans_neg_choice])
                            samples.append([query_idx+1,set_ans,set_ans_sim,ans_neg_choice])
                    else:
                        continue

            queries.append("Question: " + question['Question'])
            queries.append("Answer: " + question['Answer'])
            query_idx += 2

In [ ]:
for n_material in tqdm(range(course_materials.shape[0])):
    material_id = course_materials["contentsid"][n_material]
    max_pages = course_materials["pages"][n_material]
    title = course_materials["title"][n_material]

    slide_types = []
    with open(f"./Agent_resources/Slide_types/pid_{course_id}_{material_id}_slide_types.txt", "r") as file:
        for line in file:
            slide_types.append(line.strip())

    non_informative = ["TITLE_SLIDE","AGENDA_SLIDE","HEADER_SLIDE","CONCLUSION_SLIDE"]
    slide_filtered = []
    slide_possible = []

    for i in range(1,max_pages+1):
        if slide_types[i-1] in non_informative:
            slide_filtered.append(i)
        else:
            slide_possible.append(i)

    for q_sample in range(10):
        try:
            importances = read_jsonl(f'./Agent_resources/Importances/pid{course_id}_{material_id}_IMPORTANCE_{q_sample}_test.jsonl')
            importances = pd.DataFrame(importances)
        except:
            raise NameError(f"Error of file name: {q_sample} sample.")
            continue

        for slide_page in slide_possible:
            if importances.loc[importances["Slide"]==slide_page].shape[0]==0:
                print(f"Error, slide not included: {slide_page}")
                continue

            responses = [slide_page]
            responses_similar = list(set(importances.loc[importances["Slide"]==slide_page]["Ref_slides"].tolist()[0]) - set(slide_filtered))
            responses_negative = list(set(slide_possible)-(set(responses) | set(responses_similar)))

            set_neg_non_title = []
            set_neg_title = []
            for rn in responses_negative:
                ans_neg = slide_index_dictionary.loc[
                                (slide_index_dictionary["material_id"]==material_id)&
                                (slide_index_dictionary["page"]==rn)&
                                (slide_index_dictionary["title"]==0)]
                set_ans_neg = ans_neg["index"].to_list()
                set_neg_non_title.extend(set_ans_neg)

                ans_neg = slide_index_dictionary.loc[
                                (slide_index_dictionary["material_id"]==material_id)&
                                (slide_index_dictionary["page"]==rn)&
                                (slide_index_dictionary["title"]==1)]
                set_ans_neg = ans_neg["index"].to_list()
                set_neg_title.extend(set_ans_neg)
            
            ans = slide_index_dictionary.loc[
                (slide_index_dictionary["material_id"]==material_id)&
                (slide_index_dictionary["page"]==slide_page)&
                (slide_index_dictionary["title"]==0)]
            set_ans_non_title = ans["index"].to_list()

            ans = slide_index_dictionary.loc[
                (slide_index_dictionary["material_id"]==material_id)&
                (slide_index_dictionary["page"]==slide_page)&
                (slide_index_dictionary["title"]==1)]
            set_ans_title = ans["index"].to_list()

            if len(responses_similar)>0:
                for rs in responses_similar:
                    ans_sim = slide_index_dictionary.loc[
                        (slide_index_dictionary["material_id"]==material_id)&
                        (slide_index_dictionary["page"]==rs)&
                        (slide_index_dictionary["title"]==0)]
                    set_ans_sim_non_title = ans_sim["index"].to_list()

                    ans_sim = slide_index_dictionary.loc[
                        (slide_index_dictionary["material_id"]==material_id)&
                        (slide_index_dictionary["page"]==rs)&
                        (slide_index_dictionary["title"]==1)]
                    set_ans_sim_title = ans_sim["index"].to_list()

                    ans_neg_choice_non_title = [random.choice(set_neg_non_title)]
                    ans_neg_choice_title = [random.choice(set_neg_title)]

                    samples.append([query_idx,set_ans_non_title,set_ans_sim_non_title,ans_neg_choice_non_title])
                    samples.append([query_idx+1,set_ans_title,set_ans_sim_title,ans_neg_choice_title])
                    samples.append([query_idx+2,set_ans_title,set_ans_sim_title,ans_neg_choice_title])
            else:
                continue


            queries.append("Title: " + importances.loc[importances["Slide"]==slide_page]["Title"].tolist()[0])
            queries.append("Importance: " + importances.loc[importances["Slide"]==slide_page]["Importance"].tolist()[0])
            queries.append("Key Topics: " + importances.loc[importances["Slide"]==slide_page]["Key_topics"].tolist()[0])
            query_idx += 3

In [ ]:
with open(f"./Agent_resources/Final_finetuning/pid_{course_id}_test_queries.json", "w", encoding="utf-8") as file:
    json.dump(queries, file)

In [ ]:
with open(f"./Agent_resources/Final_finetuning/pid_{course_id}_test_slides.json", "w", encoding="utf-8") as file:
    json.dump(slide_texts, file)

In [ ]:
with open(f"./Agent_resources/Final_finetuning/pid_{course_id}_dataset_test.json", "w") as file:
    json.dump(samples, file)